# Deckalization — Resolver & Retrieval Playground

Run cells top-to-bottom. This notebook lets you **see exactly what the resolver and rules
retrieval return**, so you can decide the data direction:

- **Keep all cards** (current) — the resolver prefers playable cards over tokens/art at query time, or
- **Filter "not real" cards** — drop tokens / art-series / emblems from the mirror entirely.

Sections:
1. `resolve(name)` — the full resolution ladder result
2. `rows_for(name)` — every DB row sharing a name (the duplicate / "not real" question)
3. `fuzzy_view(name)` — the full-text candidate set + rapidfuzz re-ranking the resolver sees
4. `rules_view(query)` — semantic rules retrieval
5. Batch fixtures — run a whole list at once
6. Collision inspector — how often "not real" cards collide with real names

> Needs `.env.local` (CONVEX_URL + OPENROUTER_API_KEY) and `npx convex dev` data already loaded.

In [1]:
import os
import sys
from pathlib import Path

# --- Bootstrap: make imports + .env.local work no matter where Jupyter started ---
_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "pyproject.toml").exists():
        os.chdir(_p)
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
print("project root:", Path.cwd())

import pandas as pd
from rapidfuzz import fuzz

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 60)

from agents.config import get_settings
from agents.normalize import normalize_name
from agents.resolver import resolve_card
from agents.tools.cards import (
    get_cards_by_normalized_name,
    is_playable,
    search_cards_by_name,
)
from agents.tools.rules import search_rules

_s = get_settings()
assert _s.convex_url, "CONVEX_URL missing — check .env.local"
assert _s.openrouter_api_key, "OPENROUTER_API_KEY missing — check .env.local"
print("convex:", _s.convex_url)
print("thresholds: high=%.2f  min=%.2f" % (
    _s.thresholds.resolver_high_confidence, _s.thresholds.resolver_min_confidence))


# ---------- helpers ----------

def resolve(name: str):
    """Run the full ladder and show the result + candidates."""
    r = resolve_card(name)
    print(f"query     : {r.query!r}")
    print(f"status    : {r.status}")
    print(f"method    : {r.method}")
    print(f"confidence: {r.confidence}")
    print(f"reason    : {r.reason}")
    if r.card:
        print(f"\nRESOLVED  -> {r.card['name']}  [{r.card.get('typeLine','')}]")
        print(f"            playable={is_playable(r.card)}  layout={r.card.get('layout')}"
              f"  rulings={len(r.card.get('rulings') or [])}")
        txt = (r.card.get("oracleText") or "").replace("\n", " ")
        print(f"            {txt[:200]}")
    if r.candidates:
        print("\ncandidates:")
        display(pd.DataFrame([c.model_dump() for c in r.candidates]))
    return r


def rows_for(name: str):
    """Every DB row sharing this exact (normalized) name — the 'not real' question."""
    rows = get_cards_by_normalized_name(normalize_name(name))
    if not rows:
        print(f"no exact rows for {name!r}")
        return pd.DataFrame()
    df = pd.DataFrame([{
        "name": c["name"],
        "playable": is_playable(c),
        "layout": c.get("layout"),
        "setType": c.get("setType"),
        "typeLine": c.get("typeLine"),
        "rulings": len(c.get("rulings") or []),
        "oracleText": (c.get("oracleText") or "").replace("\n", " ")[:60],
    } for c in rows]).sort_values(["playable", "rulings"], ascending=False)
    return df.reset_index(drop=True)


def fuzzy_view(name: str, limit: int = 40):
    """What the resolver's fuzzy rung sees: full-text candidates re-ranked by rapidfuzz."""
    norm = normalize_name(name)
    cands = search_cards_by_name(name, limit=limit)
    rows = []
    for c in cands:
        playable = is_playable(c)
        base = fuzz.WRatio(norm, c.get("normalizedName", "")) / 100.0
        rows.append({
            "score": round(base - (0.0 if playable else 0.08), 3),
            "raw_wratio": round(base, 3),
            "playable": playable,
            "name": c["name"],
            "typeLine": c.get("typeLine"),
            "layout": c.get("layout"),
        })
    df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    return df


def rules_view(query: str, k: int = 8):
    """Semantic rules retrieval — what the rules agent would ground on."""
    hits = search_rules(query, limit=k)
    df = pd.DataFrame([{
        "score": round(h["score"], 3),
        "rule": h["ruleNumber"],
        "section": h["section"],
        "text": h["text"].replace("\n", " ")[:140],
    } for h in hits])
    return df


print("\nready -> try: resolve('Lightnig Bolt'),  rows_for('Llanowar Elves'),  rules_view('deathtouch trample')")

project root: /Users/diogoazevedo/Documents/Qempire/GitHub/deckalization
convex: https://friendly-mink-500.eu-west-1.convex.cloud
thresholds: high=0.92  min=0.60

ready -> try: resolve('Lightnig Bolt'),  rows_for('Llanowar Elves'),  rules_view('deathtouch trample')


## 1. Resolve a card name

Change the string and re-run. Try a clean name, a typo, a nickname, a rules concept, and gibberish:
`'Sol Ring'`, `'Lightnig Bolt'`, `'Bob'`, `'Treasure token'`, `'Qwertyuiop notacard'`.

In [ ]:
r = resolve("Lightnig Bolt")

query     : 'Lightnig Bolt'
status    : resolved
method    : fuzzy
confidence: 0.963
reason    : Confident fuzzy name match.

RESOLVED  -> Lightning Bolt  [Instant]
            playable=True  layout=normal  rulings=0
            Lightning Bolt deals 3 damage to any target.


## 2. The "not real" cards question

`rows_for(name)` shows **every** row that shares an exact name. This is the core of your
decision: how many tokens / art-series / emblems sit next to the real card, and does the
resolver's canonical pick (top playable row) stay correct while they're present?

Good names to inspect: `'Llanowar Elves'`, `'Goblin'`, `'Soldier'`, `'Spirit'`, `'Ajani's Pridemate'`, `'Lightning Bolt'`, `'Island'`.

In [5]:
rows_for("Lightnig Bolt")

no exact rows for 'Lightnig Bolt'


""


## 3. Fuzzy rung — what the resolver re-ranks

`fuzzy_view(name)` shows the full-text candidate set (Convex BM25) re-ranked by rapidfuzz.
`score` is the resolver's effective score (raw WRatio minus a 0.08 penalty for non-playable
rows). Useful for seeing *why* a typo did or didn't resolve confidently.

In [4]:
fuzzy_view("Lightnig Bolt").head(10)

,score,raw_wratio,playable,name,typeLine,layout
0,0.963,0.963,True,Lightning Bolt,Instant,normal
1,0.855,0.855,True,Tax Bolt,Instant,normal
2,0.855,0.855,True,Emeritus of Conflict // Lightning Bolt,Creature — Human Wizard // Instant,prepare
3,0.855,0.855,True,"Black Bolt, Inhuman King",Legendary Creature — Inhuman Noble Hero,normal
4,0.775,0.855,False,Flame-Blessed Bolt // Flame-Blessed Bolt,Card // Card,art_series
5,0.775,0.855,False,Lightning Bolt // Lightning Bolt,Card // Card,art_series
6,0.667,0.667,True,Obliterating Bolt,Sorcery,normal
7,0.640,0.640,True,Guiding Bolt,Instant,normal
8,0.636,0.636,True,Twin Bolt,Instant,normal
9,0.636,0.636,True,Rift Bolt,Sorcery,normal


## 4. Rules retrieval

`rules_view(query)` runs the semantic vector search over the Comprehensive Rules. Try:
`'how is combat damage assigned with multiple blockers'`, `'deathtouch and trample'`,
`'what happens when a commander would change zones'`, `'state-based actions for zero loyalty'`.

In [5]:
rules_view("how is combat damage assigned when a creature is blocked by multiple creatures")

,score,rule,section,text
0,0.711,510.1d,510. Combat Damage Step,A blocking creature assigns combat damage to the creatures it’s blocking. If...
1,0.684,510.1c,510. Combat Damage Step,A blocked creature assigns its combat damage to the creatures blocking it. I...
2,0.641,510.1b,510. Combat Damage Step,"An unblocked creature assigns its combat damage to the player, planeswalker,..."
3,0.631,510.1,510. Combat Damage Step,"First, the active player announces how each attacking creature assigns its c..."
4,0.624,510.1a,510. Combat Damage Step,Each attacking creature and each blocking creature assigns combat damage equ...
5,0.579,510.4,510. Combat Damage Step,If at least one attacking or blocking creature has first strike (see rule 70...
6,0.576,509.1h,509. Declare Blockers Step,An attacking creature with one or more creatures declared as blockers for it...
7,0.574,510.1e,510. Combat Damage Step,Once a player has assigned combat damage from each attacking or blocking cre...


## 5. Batch fixtures

Run a whole list at once to eyeball the resolver's behaviour. Edit `FIXTURES` freely.

In [6]:
FIXTURES = [
    "Lightning Bolt",        # clean
    "Lightnig Bolt",         # typo
    "jace the mind sculptr", # typo + casing
    "Bob",                   # nickname -> Dark Confidant
    "Tarmo",                 # nickname -> Tarmogoyf
    "Sol Ring",              # clean
    "Llanowar Elves",        # token collision
    "Treasure token",        # rules concept
    "The Monarch",           # rules concept
    "Qwertyuiop notacard",   # gibberish -> not_found
]

batch = []
for q in FIXTURES:
    r = resolve_card(q)
    batch.append({
        "query": q,
        "status": r.status,
        "method": r.method,
        "conf": r.confidence,
        "resolved_to": r.card["name"] if r.card else None,
        "playable": is_playable(r.card) if r.card else None,
        "candidates": len(r.candidates),
    })
pd.DataFrame(batch)

,query,status,method,conf,resolved_to,playable,candidates
0,Lightning Bolt,resolved,exact,1.000,Lightning Bolt,True,0
1,Lightnig Bolt,resolved,fuzzy,0.963,Lightning Bolt,True,0
2,jace the mind sculptr,resolved,fuzzy,0.977,"Jace, the Mind Sculptor",True,0
3,Bob,resolved,alias,0.970,Dark Confidant,True,0
4,Tarmo,resolved,alias,0.970,Tarmogoyf,False,0
5,Sol Ring,resolved,exact,1.000,Sol Ring,True,0
6,Llanowar Elves,resolved,exact,0.950,Llanowar Elves,True,0
7,Treasure token,rules_concept,concept,0.000,NaN,None,0
8,The Monarch,rules_concept,concept,0.000,NaN,None,0
9,Qwertyuiop notacard,not_found,none,0.000,NaN,None,0


## 6. Collision inspector — keep-all vs filter

For a curated set of names that famously collide with tokens / art cards, this shows how many
rows are playable vs not, and whether the resolver still picks a playable card. If the
`resolved_playable` column is always `True`, **keep-all** is working and filtering is optional.
If you see wrong picks or `False`, that argues for **filtering "not real" cards**.

In [7]:
COLLISION_NAMES = [
    "Llanowar Elves", "Goblin", "Soldier", "Spirit", "Zombie", "Elf Warrior",
    "Treasure", "Clue", "Food", "Ajani's Pridemate", "Lightning Bolt",
    "Island", "Forest", "Serra Angel", "Llanowar Mentor",
]

report = []
for name in COLLISION_NAMES:
    rows = get_cards_by_normalized_name(normalize_name(name))
    if not rows:
        report.append({"name": name, "rows": 0, "playable": 0, "not_real": 0,
                       "resolved_to": None, "resolved_playable": None})
        continue
    playable = [c for c in rows if is_playable(c)]
    r = resolve_card(name)
    report.append({
        "name": name,
        "rows": len(rows),
        "playable": len(playable),
        "not_real": len(rows) - len(playable),
        "resolved_to": r.card["name"] if r.card else None,
        "resolved_playable": is_playable(r.card) if r.card else None,
        "status": r.status,
    })
pd.DataFrame(report)

,name,rows,playable,not_real,resolved_to,resolved_playable,status
0,Llanowar Elves,2,1,1,Llanowar Elves,True,resolved
1,Goblin,7,1,6,________ Goblin,True,resolved
2,Soldier,13,0,13,Soldier,False,resolved
3,Spirit,23,0,23,Spirit,False,resolved
4,Zombie,8,0,8,Zombie,False,resolved
5,Elf Warrior,2,0,2,Elf Warrior,False,resolved
6,Treasure,2,0,2,NaN,None,rules_concept
7,Clue,1,0,1,NaN,None,rules_concept
8,Food,1,0,1,NaN,None,rules_concept
9,Ajani's Pridemate,2,1,1,Ajani's Pridemate,True,resolved
